# **Model training**
#### *so it is regression problem  and we need to check all regressors of classical ml, and to see which algorithm perfom well on current data and that will be our base model and try to improve its performance by help of hyber perameter tunning*

In [ ]:
# importing all necessary library 
import pandas as pd
import numpy as np
#import libraries for model training from sickit-learn 
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
# other classifier
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
# import performance metrics
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
# import warning, such that we have not receive any waning
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# now import data from data folder present in csv format
df = pd.read_csv('Data/students.csv')
df.head()

Here our main aim is how build end to end ml project from data ingestion to deployment, so previously we made new columns, we can make one of them as target column or from existing let we want to find writing score on the bases of math and reading score 

## **1 split data into X and y variable**

In [ ]:
X = df.drop('writing_score', axis = 1)
y = df['writing_score']
print(X.shape[1])


so we have seven features now in X

### **changing categorical columns into numeric and numerical column into standard scale using column transformer**

In [ ]:
cate = X.select_dtypes(include='str').columns
num = X.select_dtypes(exclude='str').columns
print(cate)
print(num)


In [ ]:
#using column transformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
onh = OneHotEncoder(drop='first',sparse_output=False)
stscale = StandardScaler()
preprocessing = ColumnTransformer(
    [
    ("OneHotEncoding",onh,cate),
    ("StandardScaler",stscale,num)
],remainder='passthrough'
)

to avoid data lakage we first split the data(X and y) into train and test then apply column transformation

In [ ]:
#spliting the data intot train and test
from sklearn.model_selection import train_test_split

In [ ]:
x_train,x_test,y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=42)

In [ ]:
#now apply transformation
X_train = preprocessing.fit_transform(x_train)
X_test = preprocessing.transform(x_test)

In [ ]:
#let check the shape of both
print(X_train.shape)
print(X_test.shape)

## **Model Training**
### *selecting best model*

In [ ]:
# make function to check the performance of models
def evaluate_model(true, predicted):
    mae = mean_absolute_error(true, predicted)
    mse = mean_squared_error(true, predicted)
    rmse = np.sqrt(mean_squared_error(true, predicted))
    r2_square = r2_score(true, predicted)
    return mae, rmse, r2_square

In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Lasso": Lasso(),
    "Ridge": Ridge(),
    "K-Neighbors Regressor": KNeighborsRegressor(),
    "Decision Tree": DecisionTreeRegressor(),
    "Random Forest Regressor": RandomForestRegressor(),
    "XGBRegressor": XGBRegressor(), 
    "CatBoosting Regressor": CatBoostRegressor(verbose=False),
    "AdaBoost Regressor": AdaBoostRegressor()
}
model_list = []
r2_list =[]

for i in range(len(list(models))):
    model = list(models.values())[i]
    model.fit(X_train, y_train) # Train model

    # Make predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    # Evaluate Train and Test dataset
    model_train_mae , model_train_rmse, model_train_r2 = evaluate_model(y_train, y_train_pred)

    model_test_mae , model_test_rmse, model_test_r2 = evaluate_model(y_test, y_test_pred)

    
    print(list(models.keys())[i])
    model_list.append(list(models.keys())[i])
    
    print('Model performance for Training set')
    print("- Root Mean Squared Error: {:.4f}".format(model_train_rmse))
    print("- Mean Absolute Error: {:.4f}".format(model_train_mae))
    print("- R2 Score: {:.4f}".format(model_train_r2))

    print('----------------------------------')
    
    print('Model performance for Test set')
    print("- Root Mean Squared Error: {:.4f}".format(model_test_rmse))
    print("- Mean Absolute Error: {:.4f}".format(model_test_mae))
    print("- R2 Score: {:.4f}".format(model_test_r2))
    r2_list.append(model_test_r2)
    
    print('='*35)
    print('\n')

### *Results on tests data of each model*

In [ ]:
pd.DataFrame(list(zip(model_list, r2_list)), columns=['Model Name', 'R2_Score']).sort_values(by=["R2_Score"],ascending=False)

## performance on test data is best among all is only linear and Ridge

In [ ]:
# find the accuracy of linear regression models 
lin_model = LinearRegression(fit_intercept=True)
lin_model = lin_model.fit(X_train, y_train)
y_pred = lin_model.predict(X_test)
score = r2_score(y_test, y_pred)*100
print(" Accuracy of the model is %.2f" %score)

In [ ]:
import os
os.makedirs('figures', exist_ok=True)

In [ ]:
#plot y_pred and y_test
import matplotlib
import matplotlib.pyplot as plt
matplotlib.use('Agg')
plt.scatter(y_test,y_pred)
plt.xlabel('Actual')
plt.ylabel('Predicted')
plt.savefig('figures/scatter_plot.png', bbox_inches='tight')
plt.show()
plt.close('all')

In [ ]:
import seaborn as sns
plt.figure()
sns.regplot(x=y_test, y=y_pred, ci=None, color='red')
plt.savefig('figures/reg_plot.png', bbox_inches='tight')
plt.close('all')

In [ ]:
# now finding the difference between actual and predicted 
pred_df=pd.DataFrame({'Actual Value':y_test,'Predicted Value':y_pred,'Difference':y_test-y_pred})
pred_df

In [ ]:
import json
with open('model_training.ipynb', 'r') as f:
    nb = json.load(f)
total = sum(len(json.dumps(c.get('outputs',[]))) for c in nb['cells'] if c['cell_type']=='code')
print(f'Total output size: {total/1024:.1f} KB')

In [ ]:
import json

with open('model_training.ipynb', 'r') as f:
    nb = json.load(f)

for i, cell in enumerate(nb['cells']):
    if cell['cell_type'] == 'code':
        output_str = json.dumps(cell.get('outputs', []))
        size = len(output_str)
        if size > 500:
            print(f'Cell {i}: {size/1024:.1f} KB | {str(cell["source"])[:60]}')